# Fort Worth CLT Site Scoring — Data Pipeline
This notebook pulls all data sources together into one merged CSV ready for scoring.

**Run order:**
1. Cell 1 — Install dependencies
2. Cell 2 — Load your existing CSVs (already filtered to Fort Worth)
3. Cell 3 — CDC PLACES health data
4. Cell 4 — HUD QCT / Opportunity Zone designations
5. Cell 5 — Walk Score API (requires free API key)
6. Cell 6 — Fort Worth Open Data (crime + parks)
7. Cell 7 — Tarrant Appraisal District parcel data
8. Cell 8 — FEMA flood zones (requires geopandas)
9. Cell 9 — Manual scoring CSV
10. Cell 10 — Final merge & export

In [2]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
# Run once, then restart kernel
%pip install pandas requests geopandas openpyxl shapely

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ── CELL 2: Load your existing Fort Worth CSVs ───────────────────────────────
import pandas as pd
import numpy as np
import os
from pathlib import Path

_raw = Path.cwd()
DATA_DIR = str(_raw / "data") if (_raw / "data").is_dir() else str(_raw.parent / "data")

def load(filename):
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path, low_memory=False)
    df['GEOID'] = df['GEOID'].astype(str).str.strip()
    return df

# Load the pre-filtered Fort Worth files you already have
chas    = load('fw_chas.csv')
housing = load('fw_housing.csv')
socio   = load('fw_socioeconomic.csv')
afford  = load('fw_affordability.csv')
lowmod  = load('fw_lowmod.csv')

# Pull key columns from each and rename for clarity
price = housing[[
    'GEOID',
    'B25097EST1',   # median home value
    'B25097EST2',   # median home value (owner)
    'B25058EST1',   # median gross rent
    'B25037EST3',   # median year built
]].rename(columns={
    'B25097EST1': 'median_home_value',
    'B25097EST2': 'median_home_value_owner',
    'B25058EST1': 'median_gross_rent',
    'B25037EST3': 'median_year_built',
})

aff = afford[[
    'GEOID',
    'median_hh_income',
    'median_smoc_mortgage',
    'median_gross_rent',
    'avg_h_cost',
    'hh1_ht', 'hh2_ht', 'hh3_ht',
    'hh1_pctile_all', 'hh2_pctile_all', 'hh3_pctile_all',
]].rename(columns={
    'median_smoc_mortgage':  'monthly_owner_cost',
    'avg_h_cost':            'avg_housing_cost',
    'hh1_ht':                'ht_cost_very_low_income',
    'hh2_ht':                'ht_cost_low_income',
    'hh3_ht':                'ht_cost_moderate_income',
    'hh1_pctile_all':        'affordability_pctile_very_low',
    'hh2_pctile_all':        'affordability_pctile_low',
    'hh3_pctile_all':        'affordability_pctile_moderate',
})

lmi = lowmod[[
    'GEOID', 'LOWMOD', 'LOWMODUNIV', 'LOWMODPCT'
]].rename(columns={
    'LOWMOD':     'lmi_population',
    'LOWMODUNIV': 'total_population',
    'LOWMODPCT':  'lmi_pct',
})

burden = chas[[
    'GEOID',
    'T8_LE30', 'T8_LE50', 'T8_LE80',
    'T8_LE30_CB', 'T8_LE50_CB', 'T8_LE80_CB',
    'AFF_AVAIL_30_R', 'AFF_AVAIL_50_R',
]].rename(columns={
    'T8_LE30':        'hh_at_30pct_ami',
    'T8_LE50':        'hh_at_50pct_ami',
    'T8_LE80':        'hh_at_80pct_ami',
    'T8_LE30_CB':     'cost_burdened_30ami',
    'T8_LE50_CB':     'cost_burdened_50ami',
    'T8_LE80_CB':     'cost_burdened_80ami',
    'AFF_AVAIL_30_R': 'affordable_units_30ami',
    'AFF_AVAIL_50_R': 'affordable_units_50ami',
})

se = socio[[
    'GEOID',
    'B19013EST1',   # median household income
    'B17021EST2',   # persons in poverty
    'B17021EST2_PCT',
    'B23001_UE_PCT',  # unemployment rate
]].rename(columns={
    'B19013EST1':     'median_income',
    'B17021EST2':     'poverty_count',
    'B17021EST2_PCT': 'poverty_pct',
    'B23001_UE_PCT':  'unemployment_pct',
})

# Base merge
base = (price
        .merge(aff,    on='GEOID', how='outer')
        .merge(lmi,    on='GEOID', how='outer')
        .merge(burden, on='GEOID', how='outer')
        .merge(se,     on='GEOID', how='outer'))

print(f"Base dataset: {base.shape[0]} tracts, {base.shape[1]} columns")
base.head(2)

Base dataset: 530 tracts, 30 columns


,GEOID,median_home_value,median_home_value_owner,median_gross_rent_x,median_year_built,median_hh_income,monthly_owner_cost,median_gross_rent_y,avg_housing_cost,ht_cost_very_low_income,...,hh_at_80pct_ami,cost_burdened_30ami,cost_burdened_50ami,cost_burdened_80ami,affordable_units_30ami,affordable_units_50ami,median_income,poverty_count,poverty_pct,unemployment_pct
0,48439110205,233600.0,234100.0,1125.0,1992.0,NaN,NaN,NaN,NaN,NaN,...,730.0,60.0,220.0,600.0,0.0,0.0,69662.0,363.0,7.12,2.00
1,48439111315,227800.0,239500.0,2140.0,2006.0,NaN,NaN,NaN,NaN,NaN,...,440.0,15.0,155.0,320.0,0.0,0.0,70497.0,641.0,14.64,11.75


In [14]:
# ── CELL 3: CDC PLACES — health outcomes by tract ────────────────────────────
import csv

PLACES_FILE = os.path.join(DATA_DIR, 'PLACES_tract.csv')

if os.path.exists(PLACES_FILE):
    places = pd.read_csv(PLACES_FILE,
                         engine='python',
                         on_bad_lines='skip')
    places['GEOID'] = places['LocationName'].astype(str).str.strip()

    fw_places = places[
        (places['StateAbbr'] == 'TX') &
        (places['GEOID'].str.startswith('48439'))
    ].copy()

    health_measures = [
        'CASTHMA',
        'DEPRESSION',
        'OBESITY',
        'DIABETES',
        'BPHIGH',
        'CHECKUP',
    ]

    fw_places_pivot = (
        fw_places[fw_places['MeasureId'].isin(health_measures)]
        [['GEOID', 'MeasureId', 'Data_Value']]
        .pivot_table(index='GEOID', columns='MeasureId', values='Data_Value')
        .reset_index()
    )
    fw_places_pivot.columns.name = None
    fw_places_pivot = fw_places_pivot.rename(columns={
        'CASTHMA':    'pct_asthma',
        'DEPRESSION': 'pct_depression',
        'OBESITY':    'pct_obesity',
        'DIABETES':   'pct_diabetes',
        'BPHIGH':     'pct_high_bp',
        'CHECKUP':    'pct_annual_checkup',
    })

    base = base.merge(fw_places_pivot, on='GEOID', how='left')
    print(f"CDC PLACES joined: {fw_places_pivot.shape[0]} tracts")

else:
    print("PLACES_tract.csv not found — skipping. Download from data.cdc.gov and re-run.")
    for col in ['pct_asthma','pct_depression','pct_obesity','pct_diabetes','pct_high_bp','pct_annual_checkup']:
        base[col] = np.nan

CDC PLACES joined: 447 tracts


In [3]:
# ── CELL 4: HUD QCT & Opportunity Zone designations ─────────────────────────
# QCT list: https://www.huduser.gov/portal/datasets/qct.html  (download Excel)
# OZ list:  https://www.cdfifund.gov/opportunity-zones  (CSV from Treasury)
# Save as: HUD_QCT.xlsx and OZ_tracts.csv in your DATA_DIR

QCT_FILE = os.path.join(DATA_DIR, 'HUD_QCT.xlsx')
OZ_FILE  = os.path.join(DATA_DIR, 'OZ_tracts.csv')

if os.path.exists(QCT_FILE):
    qct = pd.read_excel(QCT_FILE, dtype={'GEOID': str})
    qct['GEOID'] = qct['GEOID'].str.strip()
    fw_qct = qct[qct['GEOID'].str.startswith('48439')]
    base['is_qct'] = base['GEOID'].isin(fw_qct['GEOID']).astype(int)
    print(f"QCT: {base['is_qct'].sum()} Fort Worth tracts flagged as QCT")
else:
    print("HUD_QCT.xlsx not found — skipping.")
    base['is_qct'] = np.nan

if os.path.exists(OZ_FILE):
    oz = pd.read_csv(OZ_FILE, dtype={'geoid': str})
    oz['geoid'] = oz['geoid'].str.strip()
    fw_oz = oz[oz['geoid'].str.startswith('48439')]
    base['is_opportunity_zone'] = base['GEOID'].isin(fw_oz['geoid']).astype(int)
    print(f"OZ: {base['is_opportunity_zone'].sum()} Fort Worth tracts are Opportunity Zones")
else:
    print("OZ_tracts.csv not found — skipping.")
    base['is_opportunity_zone'] = np.nan

HUD_QCT.xlsx not found — skipping.
OZ_tracts.csv not found — skipping.


In [4]:
# ── CELL 5: Walk Score API ───────────────────────────────────────────────────
# Sign up free at: https://www.walkscore.com/professional/api.php
# Paste your key below. Free tier = 5,000 calls.
# This cell pulls walk/transit/bike scores for each tract centroid.

import requests, time

WALKSCORE_API_KEY = 'YOUR_KEY_HERE'  # <-- paste your key

# Tract centroids — we compute them from GEOID using Census Tiger data
# OR if you have lat/lon columns already, use those instead.
# Here we fetch centroids from the Census Geocoder for each GEOID.

def get_tract_centroid(geoid):
    """Get lat/lon centroid for a census tract GEOID via Census TIGERweb."""
    state  = geoid[:2]
    county = geoid[2:5]
    tract  = geoid[5:]
    url = (f"https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
           f"Tracts_Blocks/MapServer/0/query"
           f"?where=STATE='{state}'+AND+COUNTY='{county}'+AND+TRACT='{tract}'"
           f"&outFields=CENTLAT,CENTLON&f=json")
    try:
        r = requests.get(url, timeout=10)
        feat = r.json()['features'][0]['attributes']
        return float(feat['CENTLAT']), float(feat['CENTLON'])
    except:
        return None, None

def get_walk_score(lat, lon, geoid):
    """Call Walk Score API for a given lat/lon."""
    if lat is None:
        return None, None, None
    try:
        r = requests.get('https://api.walkscore.com/score', params={
            'lat': lat, 'lon': lon,
            'address': f'Census Tract {geoid}',
            'transit': 1, 'bike': 1,
            'wsapikey': WALKSCORE_API_KEY,
            'format': 'json'
        }, timeout=10)
        data = r.json()
        walk    = data.get('walkscore')
        transit = data.get('transit', {}).get('score') if data.get('transit') else None
        bike    = data.get('bike', {}).get('score') if data.get('bike') else None
        return walk, transit, bike
    except:
        return None, None, None

if WALKSCORE_API_KEY != 'YOUR_KEY_HERE':
    walk_results = []
    for geoid in base['GEOID']:
        lat, lon = get_tract_centroid(geoid)
        walk, transit, bike = get_walk_score(lat, lon, geoid)
        walk_results.append({'GEOID': geoid, 'walk_score': walk,
                             'transit_score': transit, 'bike_score': bike})
        time.sleep(0.1)  # be gentle with the API

    walk_df = pd.DataFrame(walk_results)
    base = base.merge(walk_df, on='GEOID', how='left')
    print(f"Walk Score: scores retrieved for {walk_df['walk_score'].notna().sum()} tracts")
else:
    print("Walk Score API key not set — skipping. Add your key and re-run.")
    base['walk_score']    = np.nan
    base['transit_score'] = np.nan
    base['bike_score']    = np.nan

Walk Score API key not set — skipping. Add your key and re-run.


In [5]:
# ── CELL 6: Fort Worth Open Data — Crime & Parks ─────────────────────────────
# Crime:  https://data.fortworthtexas.gov  → search "crime" → download CSV
# Parks:  https://data.fortworthtexas.gov  → search "parks" → download CSV
# Save as: FW_crime.csv and FW_parks.csv in your DATA_DIR
#
# Both files need lat/lon columns to assign tracts.
# We use the Census Geocoder to convert coordinates → GEOID.

CRIME_FILE = os.path.join(DATA_DIR, 'FW_crime.csv')
PARKS_FILE = os.path.join(DATA_DIR, 'FW_parks.csv')

def coords_to_tract(lat, lon):
    """Convert lat/lon to census tract GEOID using Census Geocoder."""
    try:
        r = requests.get(
            'https://geocoding.geo.census.gov/geocoder/geographies/coordinates',
            params={'x': lon, 'y': lat,
                    'benchmark': 'Public_AR_Current',
                    'vintage': 'Current_Current',
                    'format': 'json'},
            timeout=10
        )
        tracts = r.json()['result']['geographies'].get('Census Tracts', [])
        return tracts[0]['GEOID'] if tracts else None
    except:
        return None

# --- Crime ---
if os.path.exists(CRIME_FILE):
    crime = pd.read_csv(CRIME_FILE, low_memory=False)
    print(f"Crime columns: {list(crime.columns[:10])}")
    # Adjust column names below to match what FW Open Data actually provides:
    LAT_COL = 'Latitude'   # <-- check your file's actual column name
    LON_COL = 'Longitude'  # <-- check your file's actual column name

    crime = crime.dropna(subset=[LAT_COL, LON_COL])
    # Assign tracts (sample: geocode first 5000 rows to save API calls)
    crime_sample = crime.head(5000).copy()
    crime_sample['GEOID'] = crime_sample.apply(
        lambda r: coords_to_tract(r[LAT_COL], r[LON_COL]), axis=1)

    crime_by_tract = (
        crime_sample.groupby('GEOID').size()
        .reset_index(name='crime_incidents')
    )
    base = base.merge(crime_by_tract, on='GEOID', how='left')
    print(f"Crime: aggregated to {crime_by_tract.shape[0]} tracts")
else:
    print("FW_crime.csv not found — skipping.")
    base['crime_incidents'] = np.nan

# --- Parks ---
if os.path.exists(PARKS_FILE):
    parks = pd.read_csv(PARKS_FILE, low_memory=False)
    print(f"Parks columns: {list(parks.columns[:10])}")
    LAT_COL = 'Latitude'   # <-- check your file's actual column name
    LON_COL = 'Longitude'  # <-- check your file's actual column name

    parks = parks.dropna(subset=[LAT_COL, LON_COL])
    parks['GEOID'] = parks.apply(
        lambda r: coords_to_tract(r[LAT_COL], r[LON_COL]), axis=1)

    parks_by_tract = (
        parks.groupby('GEOID').size()
        .reset_index(name='parks_count')
    )
    base = base.merge(parks_by_tract, on='GEOID', how='left')
    print(f"Parks: aggregated to {parks_by_tract.shape[0]} tracts")
else:
    print("FW_parks.csv not found — skipping.")
    base['parks_count'] = np.nan

FW_crime.csv not found — skipping.
FW_parks.csv not found — skipping.


In [ ]:
# ── CELL 7: Tarrant Appraisal District — parcel data ─────────────────────────
# Download from: https://www.tad.org  → Data → Property Data Export
# Save as: TAD_parcels.csv in your DATA_DIR
#
# Key fields we want: improvement_value, land_value, situs_city, lat, lon
# Column names vary by export year — print columns and adjust below.

TAD_FILE = os.path.join(DATA_DIR, 'TAD_parcels.csv')

if os.path.exists(TAD_FILE):
    parcels = pd.read_csv(TAD_FILE, low_memory=False)
    print(f"TAD columns: {list(parcels.columns[:15])}")

    # Adjust column names to match TAD export:
    CITY_COL   = 'situs_city'        # city name column
    IMP_COL    = 'improvement_value' # building/improvement value
    LAND_COL   = 'land_value'        # land value
    LAT_COL    = 'latitude'
    LON_COL    = 'longitude'

    fw_parcels = parcels[
        parcels[CITY_COL].str.upper().str.contains('FORT WORTH', na=False)
    ].copy()

    # Flag vacant lots (no improvement value) and distressed (very low value)
    fw_parcels['is_vacant']     = (fw_parcels[IMP_COL].fillna(0) == 0).astype(int)
    fw_parcels['total_value']   = fw_parcels[IMP_COL].fillna(0) + fw_parcels[LAND_COL].fillna(0)

    # Assign census tract via geocoder (sample to limit API calls)
    fw_parcels = fw_parcels.dropna(subset=[LAT_COL, LON_COL]).head(10000)
    fw_parcels['GEOID'] = fw_parcels.apply(
        lambda r: coords_to_tract(r[LAT_COL], r[LON_COL]), axis=1)

    parcel_by_tract = fw_parcels.groupby('GEOID').agg(
        vacant_lot_pct    = ('is_vacant', 'mean'),
        median_land_value = (LAND_COL, 'median'),
        parcel_count      = ('is_vacant', 'count'),
    ).reset_index()

    base = base.merge(parcel_by_tract, on='GEOID', how='left')
    print(f"TAD parcels: aggregated to {parcel_by_tract.shape[0]} tracts")
else:
    print("TAD_parcels.csv not found — skipping.")
    base['vacant_lot_pct']    = np.nan
    base['median_land_value'] = np.nan
    base['parcel_count']      = np.nan

In [ ]:
# ── CELL 8: FEMA Flood Zones ─────────────────────────────────────────────────
# Download Tarrant County NFHL shapefile from: https://msc.fema.gov/portal/home
# Search: Tarrant County TX → download FIRM shapefile
# Also download census tract shapefile:
#   https://www.census.gov/cgi-bin/geo/shapefiles/index.php
#   → Tracts → Texas
# Save shapefiles to your DATA_DIR

try:
    import geopandas as gpd

    FLOOD_SHP  = os.path.join(DATA_DIR, 'tarrant_NFHL', 'S_FLD_HAZ_AR.shp')
    TRACTS_SHP = os.path.join(DATA_DIR, 'tl_2023_48_tract.shp')

    if os.path.exists(FLOOD_SHP) and os.path.exists(TRACTS_SHP):
        tracts_gdf = gpd.read_file(TRACTS_SHP)
        tracts_gdf = tracts_gdf[tracts_gdf['COUNTYFP'] == '439'].copy()  # Tarrant County
        tracts_gdf['GEOID'] = tracts_gdf['GEOID'].astype(str)

        flood_gdf  = gpd.read_file(FLOOD_SHP)
        flood_gdf  = flood_gdf.to_crs(tracts_gdf.crs)

        # Flag high-risk flood zones (AE, A, AO, AH = high risk)
        high_risk = flood_gdf[flood_gdf['FLD_ZONE'].str.startswith('A', na=False)]

        # Calculate % of each tract area that overlaps high-risk flood zone
        tracts_gdf['tract_area'] = tracts_gdf.geometry.area
        overlay = gpd.overlay(tracts_gdf[['GEOID','geometry','tract_area']],
                              high_risk[['geometry']], how='intersection')
        overlay['flood_area'] = overlay.geometry.area
        flood_by_tract = overlay.groupby('GEOID')['flood_area'].sum().reset_index()
        flood_by_tract = flood_by_tract.merge(
            tracts_gdf[['GEOID','tract_area']], on='GEOID')
        flood_by_tract['flood_zone_pct'] = (
            flood_by_tract['flood_area'] / flood_by_tract['tract_area'] * 100
        ).round(2)

        base = base.merge(flood_by_tract[['GEOID','flood_zone_pct']], on='GEOID', how='left')
        base['flood_zone_pct'] = base['flood_zone_pct'].fillna(0)
        print(f"FEMA flood zones joined. {(base['flood_zone_pct']>10).sum()} tracts >10% in flood zone.")
    else:
        print("Flood or tract shapefiles not found — skipping.")
        base['flood_zone_pct'] = np.nan

except ImportError:
    print("geopandas not installed. Run: pip install geopandas and re-run this cell.")
    base['flood_zone_pct'] = np.nan

In [ ]:
# ── CELL 9: Manual scoring CSV ───────────────────────────────────────────────
# Fill this in for each tract you are actively evaluating.
# Score each field 1–5 (5 = best / most favorable for CLT development)
#
# This cell creates a blank template if none exists yet,
# or loads your existing one if you've already filled it in.

MANUAL_FILE = os.path.join(DATA_DIR, 'manual_scores.csv')

if not os.path.exists(MANUAL_FILE):
    # Create blank template from your tract list
    manual_template = pd.DataFrame({'GEOID': base['GEOID'].unique()})
    manual_template['site_condition_score']      = np.nan  # criterion 7
    manual_template['dev_plan_score']            = np.nan  # criterion 8
    manual_template['community_sentiment_score'] = np.nan  # criterion 13
    manual_template['political_will_score']      = np.nan  # criterion 10
    manual_template['notes']                     = ''
    manual_template.to_csv(MANUAL_FILE, index=False)
    print(f"Template created: {MANUAL_FILE}")
    print("Open it, fill in scores for sites you're evaluating, then re-run this cell.")
else:
    manual = pd.read_csv(MANUAL_FILE, dtype={'GEOID': str})
    manual['GEOID'] = manual['GEOID'].str.strip()
    base = base.merge(
        manual.drop(columns=['notes'], errors='ignore'),
        on='GEOID', how='left'
    )
    filled = manual[['site_condition_score','dev_plan_score',
                     'community_sentiment_score']].notna().any(axis=1).sum()
    print(f"Manual scores loaded: {filled} tracts have at least one manual score")

In [ ]:
# ── CELL 10: Final merge, clean up & export ───────────────────────────────────

# Drop duplicate columns from multi-source merges
base = base.loc[:, ~base.columns.duplicated()]

# Summary of coverage
print("=== Final dataset coverage ===")
print(f"Total tracts:  {len(base)}")
print(f"Total columns: {base.shape[1]}")
print("\nNon-null counts per key field:")
key_cols = [
    'median_home_value', 'median_gross_rent',
    'lmi_pct', 'cost_burdened_50ami',
    'median_hh_income', 'ht_cost_low_income',
    'pct_asthma', 'is_qct', 'is_opportunity_zone',
    'walk_score', 'transit_score',
    'crime_incidents', 'parks_count',
    'vacant_lot_pct', 'median_land_value',
    'flood_zone_pct',
    'site_condition_score', 'dev_plan_score', 'community_sentiment_score',
]
for col in key_cols:
    if col in base.columns:
        n = base[col].notna().sum()
        print(f"  {col:<35} {n:>4} / {len(base)}")
    else:
        print(f"  {col:<35} NOT YET ADDED")

# Export
OUT_PATH = os.path.join(DATA_DIR, 'fort_worth_full_pipeline.csv')
base.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")
print("Ready for scoring algorithm.")